# Introduction to LangChain, PromptTemplate (LCEL), and Jinja2 Template

**LangChain** is a framework for developing applications powered by large language models (LLMs). It simplifies chaining multiple components like prompts, LLMs, memory, and tools.

**PromptTemplate** in LangChain allows you to structure prompts with variable placeholders, making prompts dynamic and reusable.

**LCEL (LangChain Expression Language)** provides a declarative, pipeline-style syntax (like Unix pipes) to compose chains.

**Jinja2 templates** allow conditional logic and formatting inside prompt templates — ideal for dynamic prompts with customer-specific content like tone, type, or context.


In [ ]:
# !pip  install -qU cohere
# !pip install -qU openpyxl

### OCI Generative AI service LLM Access

In [ ]:
import oci
from LoadProperties import LoadProperties
properties=LoadProperties()

# use this for direct cohere model access using cohere-api-key

# YOUR_COHERE_API_KEY = ""
# Initialize the Cohere LLM
# Import LangChain components
# from langchain.llms import Cohere
# Initialize the Cohere language model
# llm = Cohere(cohere_api_key=YOUR_COHERE_API_KEY, temperature=0.7)

# OCI Generative AI service LLM Access

# Import LangChain components
from langchain_community.chat_models.oci_generative_ai import ChatOCIGenAI

# Initialize the Cohere language model
llm = ChatOCIGenAI(
      model_id='meta.llama-3.3-70b-instruct',
      service_endpoint=properties.getEndpoint(),
      compartment_id=properties.getCompartment(),auth_type='INSTANCE_PRINCIPAL',
      model_kwargs={ "max_tokens": 600},)

### Example1 PromptTemplate : Slogan for the financial product

In [ ]:
# Import LangChain components
from langchain.prompts import PromptTemplate

# Create the prompt template
prompt = PromptTemplate.from_template(
    "Act as a branding expert. Create a compelling and trustworthy slogan for the financial product line: {product_name}."
)

# Create the runnable chain
chain = prompt | llm

# Provide input
input_data = {"product_name": "WealthShield Investment Plans"}

# Run the chain
output = chain.invoke(input_data)
# print("AI-Generated Slogan:\n:", output.content)

from IPython.display import Markdown, display
display(Markdown( "AI-Generated Slogan:\n " + output.content))


### Example2.1 PromptTemplate : Banking support scenario for handling Transaction Dispute 
### Direct input 

In [ ]:

# Import LangChain components
from langchain.llms import Cohere
# Import LangChain components
from langchain.prompts import PromptTemplate

# Create the prompt template
prompt = PromptTemplate.from_template(
    """You are a customer service representative at a bank.
Write a {tone} follow-up email with the subject: "{subject}".
Use the following customer interaction summary to generate the message:\n\n{context}\n\nEmail:"""
)

chain = prompt | llm

# Example input from a real banking support scenario

email_input = {
    "tone": "empathetic and professional",
    "subject": "Update on Your Recent Transaction Dispute",
    "context": (
        "The customer reported an unauthorized debit of ₹5,000 on June 8 from their savings account. "
        "They confirmed they did not initiate the transaction. We informed them that an internal investigation has been initiated, "
        "and the resolution timeline is 3–5 business days as per bank policy."
    )
}

# Generate the email response
output = chain.invoke(email_input)

# Display the AI-generated email

# print("AI-Generated Complaint Resolution Email:\n", output.content)

from IPython.display import Markdown, display
display(Markdown( "AI-Generated Complaint Resolution Email:\n " + output.content))


### Example2.2 PromptTemplate : Banking support scenario for handling Transaction Dispute 
### Input from Excel Sheet

In [ ]:
import pandas as pd
# Import LangChain components
from langchain.llms import Cohere
# Import LangChain components
from langchain.prompts import PromptTemplate

# Load multiple inputs from Excel
df = pd.read_excel("banking_email_input_multiple.xlsx")

# Define prompt template

prompt = PromptTemplate.from_template(
    """You are a customer service representative at a bank.
Write a {tone} follow-up email with the subject: "{subject}".
Use the following customer interaction summary to generate the message:\n\n{context}\n\nEmail:"""
)

# Create LangChain pipeline
chain = prompt | llm

# Process each row and store responses
email_outputs = []
for _, row in df.iterrows():
    input_data = row.to_dict()
    email = chain.invoke(input_data)
    email_outputs.append(email)

# Add results to the dataframe and export
df["Generated_Email"] = email_outputs
output_path = "banking_email_output.xlsx"
df.to_excel(output_path, index=False)

print(f"Batch email generation completed. Output saved to: {output_path}")


### Example2.3 PromptTemplate : Banking support scenario for handling Transaction Dispute 
### Input from Excel Sheet and Send Emails to customers

In [ ]:
import pandas as pd
# Import LangChain components
from langchain.llms import Cohere
# Import LangChain components
from langchain.prompts import PromptTemplate
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
import smtplib


# SMTP email configuration (Gmail example)
SMTP_SERVER = "smtp.gmail.com"
SMTP_PORT = 587
SENDER_EMAIL = "meghatse@gmail.com"        #"your_email@gmail.com"   
SENDER_PASSWORD = "xjbplfyhwtbyznps"         # "your_app_password"  # Use an app-specific password   
RECIPIENT_EMAIL = ""      # "recipient@example.com"  # Can be dynamic per row if needed  

# === Load Excel Input ===
df = pd.read_excel("banking_email_input_multiple.xlsx")

# === Define Prompt ===

prompt = PromptTemplate.from_template(
    """You are a customer service representative at a bank.
Write a {tone} follow-up email with the subject: "{subject}".
Use the following customer interaction summary to generate the message:\n\n{context}\n\nEmail:"""
)

chain = prompt | llm

# === Generate and Send Emails ===
generated_emails = []
html_emails = []
smtp_statuses = []

for _, row in df.iterrows():
    input_data = row.to_dict()

    # Generate plain email
    # plain_email = chain.invoke(input_data)

# Invoke the language model with the filled prompt
    response = chain.invoke(input_data)
    plain_email = response.content
    generated_emails.append(plain_email)

    # Convert to HTML
    body_html = plain_email.replace("\n", "<br>")
    context_html = row["context"].replace("\n", "<br>")
    html_email = (
        f"<html><body>"
        f"<p><strong>Subject:</strong> {row['subject']}</p>"
        f"<p><strong>Tone:</strong> {row['tone']}</p>"
        f"<p><strong>Customer Context:</strong><br>{context_html}</p><hr>"
        f"<p>{body_html}</p></body></html>"
    )
    html_emails.append(html_email)

    # Prepare MIME email
    msg = MIMEMultipart("alternative")
    msg["Subject"] = row["subject"]
    msg["From"] = SENDER_EMAIL
    msg["To"] = RECIPIENT_EMAIL
    msg.attach(MIMEText(html_email, "html"))

    # Send email
    try:
        with smtplib.SMTP(SMTP_SERVER, SMTP_PORT) as server:
            server.starttls()
            server.login(SENDER_EMAIL, SENDER_PASSWORD)
            server.sendmail(SENDER_EMAIL, RECIPIENT_EMAIL, msg.as_string())
        smtp_statuses.append("Sent successfully")
    except Exception as e:
        smtp_statuses.append(f"Failed to send: {str(e)}")

# === Save Output ===
df["Generated_Email"] = generated_emails
df["HTML_Email"] = html_emails
df["SMTP_Status"] = smtp_statuses
df.to_excel("banking_email_output_with_html_smtp_sent.xlsx", index=False)

print("All emails processed. Results saved to 'banking_email_output_with_html_smtp_sent.xlsx'")


### Example3 PromptTemplate with Jinja2 formatting- Dynamic Prompt for Insurance Claim Assistance


In [ ]:
# Import LangChain components
from langchain.prompts import PromptTemplate
# Import LangChain components
from langchain.llms import Cohere


# Define a dynamic prompt to personalize claim explanations

# Jinja2-based dynamic prompt template

template_str = """
You are an AI assistant at a financial services company helping customers understand their insurance claims.

{% if customer_type == "premium" %}
Respond in a professional, detailed tone with appreciation for customer loyalty.
{% else %}
Respond in a clear, supportive tone with helpful instructions.
{% endif %}

Claim Type: {{ claim_type }}

{% if claim_type == "health" %}
Explain the health insurance claim process, required documents, and expected timeline.
{% elif claim_type == "vehicle" %}
Describe how to file a vehicle insurance claim, inspection steps, and settlement terms.
{% elif claim_type == "life" %}
Outline the steps for a life insurance claim including nominee verification and required forms.
{% else %}
Advise the customer to contact support for claim type-specific instructions.
{% endif %}
"""


# Create a PromptTemplate with Jinja2 formatting

prompt = PromptTemplate.from_template(template_str, template_format="jinja2")


# Define user-specific input for the claim type and customer type
input_variables = {
    "claim_type": "vehicle",        # Options: "health", "vehicle", "life", or others
    "customer_type": "regular"      # Options: "premium" or "regular"
}

#  Format the prompt and call the LLM

filled_prompt = prompt.format(**input_variables)
print("Filled Prompt:\n", filled_prompt)
print("=======================================================================\n")

# Invoke the language model with the filled prompt

response = llm.invoke(filled_prompt)

# print("\nAI Response:\n", response.content)

from IPython.display import Markdown, display
display(Markdown( "AI-Generated Response :\n " + response.content))

### Simulate different claim types 

In [ ]:

# Define user-specific input for the claim type and customer type

input_variables = {
    "claim_type": "health",        # Options: "health", "vehicle", "life", or others
    "customer_type": "premium"      # Options: "premium" or "regular"
}

# Format the prompt and call the LLM

filled_prompt = prompt.format(**input_variables)

print("Filled Prompt:\n", filled_prompt)
print("=======================================================================\n")


# Invoke the language model with the filled prompt

response = llm.invoke(filled_prompt)

# print("\nAI Response:\n", response.content)

from IPython.display import Markdown, display
display(Markdown( "AI-Generated Response :\n " + response.content))